In [1]:
# %load_ext autoreload
# %autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
from typing import Tuple, List, Set

# Add project root to sys.path
project_root = Path('../..').resolve()
sys.path.append(str(project_root))

# Import project modules

from src.counterfactuals.core import (
    BasisGenerator, 
    TSFeatureSchema, 
    TargetSpec, 
    GeneratorConfig, 
    LossWeights,
)
from src.counterfactuals.basis import BSplineBasis


from torch.utils.data import DataLoader

from src.data_loader.wind_turbine.turbine_data_loader import get_turbine_data_loader
from src.models.wind_turbine.anomaly_transformer.model.AnomalyTransformer import AnomalyTransformer

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device("cpu")
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


In [2]:
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# PROCESSED_DIR = project_root / "data" / "processed" / "ieee_phm"
DATA_DIR = project_root / "data" / "raw" / "wind_turbine"
DATA_DIR_TRAIN = DATA_DIR / "train"
DATA_DIR_TEST = DATA_DIR / "test"

OUTPUT_DIR = project_root / "outputs" / "wind_turbine"
SAVE_MODEL_DIR = OUTPUT_DIR / "saved_models"

# print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model save directory: {SAVE_MODEL_DIR}")

Output directory: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine
Model save directory: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/saved_models


In [4]:
# Configuration
WIN_SIZE = 144
STEP = 144
BATCH_SIZE = 64
INPUT_C = 82
OUTPUT_C = 82
E_LAYERS = 3
TEMPERATURE = 50
ANORMLY_RATIO = 1.5

# Model class mapping
MODEL_CLASSES = {
    '0': 'Generator Bearing',
    '1': 'Gearbox',
    '2': 'Transformer',
    '3': 'Hydraulic'
}

# Threshold map (from solver.py)
THRESH_MAP = {
    '0': 0.00013377491304709106,
    '1': 0.00044517662157886675,
    '2': 0.00014006024604896057,
    '3': 7.12080005177995e-05
}

DATASET_NAME = "Wind Farm A"

In [5]:
def my_kl_loss(p, q):
    res = p * (torch.log(p + 0.0001) - torch.log(q + 0.0001))
    return torch.mean(torch.sum(res, dim=-1), dim=1)

In [6]:
# filepath: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/notebooks/wind_turbine/01_load_data_and_models.ipynb
import joblib

# ===== SELECT WHICH MODEL CLASS TO ANALYZE =====
MODEL_ = '0'  # Change to '0', '1', '2', or '3'

print(f"Loading model class: {MODEL_} ({MODEL_CLASSES[MODEL_]})")

model_name = DATASET_NAME + MODEL_

# Paths
scaler_path = OUTPUT_DIR / f"model_{MODEL_}" / "scaler.joblib"
checkpoint_path = SAVE_MODEL_DIR / f"{DATASET_NAME}{MODEL_}_checkpoint1_.pth"

# Load scaler
scaler = joblib.load(scaler_path)
print(f"Scaler loaded from {scaler_path}")

# Load test data
test_loader, features, _, _ = get_turbine_data_loader(
    str(DATA_DIR_TEST), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='test', scaler=scaler
)

# Load train data (for threshold calibration reference)
train_loader, _, _, background_data = get_turbine_data_loader(
    str(DATA_DIR_TRAIN), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='train', scaler=scaler
)

# Load validation data
vali_loader, _, _, _ = get_turbine_data_loader(
    str(DATA_DIR_TRAIN), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='val', scaler=scaler
)

print(f"\nNumber of features: {len(features)}")
print(f"Features: {features[:10]}... (showing first 10)")

Loading model class: 0 (Generator Bearing)
Scaler loaded from /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/model_0/scaler.joblib
test: torch.Size([7777, 82])


/home/rdb/Documents/nirban_documents/python_programs/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


train: torch.Size([46800, 82])
val: torch.Size([5328, 82])

Number of features: 82
Features: ['status_type_id', 'sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg', 'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min', 'wind_speed_3_std', 'sensor_5_avg']... (showing first 10)


In [7]:
# Build and load model
model = AnomalyTransformer(
    win_size=WIN_SIZE,
    enc_in=INPUT_C,
    c_out=OUTPUT_C,
    e_layers=E_LAYERS
)

model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=False)
model.to(device)
model.eval()

thresh = THRESH_MAP[MODEL_]
print(f"Model loaded from: {checkpoint_path}")
print(f"Threshold for class {MODEL_} ({MODEL_CLASSES[MODEL_]}): {thresh:.10f}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model loaded from: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/saved_models/Wind Farm A0_checkpoint1_.pth
Threshold for class 0 (Generator Bearing): 0.0001337749
Model parameters: 4,918,389


# Generate Counterfactuals